In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [3]:
apps = pd.read_csv('/Users/samaniqbalansari/Downloads/googleplaystore.csv')
reviews = pd.read_csv('/Users/samaniqbalansari/Downloads/googleplaystore_user_reviews.csv')
print("Apps dataset shape:", apps.shape)
print("Reviews dataset shape:", reviews.shape)

Apps dataset shape: (10841, 13)
Reviews dataset shape: (64295, 5)


In [4]:
apps.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [5]:
apps.info()


<class 'pandas.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  str    
 1   Category        10841 non-null  str    
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  str    
 4   Size            10841 non-null  str    
 5   Installs        10841 non-null  str    
 6   Type            10840 non-null  str    
 7   Price           10841 non-null  str    
 8   Content Rating  10840 non-null  str    
 9   Genres          10841 non-null  str    
 10  Last Updated    10841 non-null  str    
 11  Current Ver     10833 non-null  str    
 12  Android Ver     10838 non-null  str    
dtypes: float64(1), str(12)
memory usage: 1.1 MB


In [6]:
apps.describe()

,Rating
count,9367.000000
mean,4.193338
std,0.537431
min,1.000000
25%,4.000000
50%,4.300000
75%,4.500000
max,19.000000


In [7]:
print("Missing values per column:")
print(apps.isnull().sum())

Missing values per column:
App                  0
Category             0
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       1
Genres               0
Last Updated         0
Current Ver          8
Android Ver          3
dtype: int64


In [8]:
print("Row 10472:")
print(apps.iloc[10472])
apps = apps.drop(apps.index[10472])
apps = apps.reset_index(drop=True)
print("\nRow dropped. New shape:",apps.shape)

Row 10472:
App               Life Made WI-Fi Touchscreen Photo Frame
Category                                              1.9
Rating                                               19.0
Reviews                                              3.0M
Size                                               1,000+
Installs                                             Free
Type                                                    0
Price                                            Everyone
Content Rating                                        NaN
Genres                                  February 11, 2018
Last Updated                                       1.0.19
Current Ver                                    4.0 and up
Android Ver                                           NaN
Name: 10472, dtype: object

Row dropped. New shape: (10840, 13)


In [9]:
print("Duplicate rows:",apps.duplicated().sum())
print("Duplicate App names:",apps.duplicated(subset='App').sum())
apps=apps.sort_values('Reviews',ascending=False)
apps=apps.drop_duplicates(subset='App',keep='first')
apps=apps.reset_index(drop=True)

print("After removing duplicates:",apps.shape)

Duplicate rows: 483
Duplicate App names: 1181
After removing duplicates: (9659, 13)


In [10]:

apps['Reviews']= pd.to_numeric(apps['Reviews'],errors='coerce')
print("Reviews dtype:",apps['Reviews'].dtype)
print("Sample values:",apps['Reviews'].head())

Reviews dtype: int64
Sample values: 0    9992
1     999
2    9975
3    9971
4    9971
Name: Reviews, dtype: int64


In [11]:
apps['Installs'] = apps['Installs'].str.replace(',', '', regex=False)
apps['Installs'] = apps['Installs'].str.replace('+', '', regex=False)
apps['Installs'] = apps['Installs'].str.strip()
apps['Installs'] = pd.to_numeric(apps['Installs'], errors='coerce')
print("Installs dtype:", apps['Installs'].dtype)
print("Sample values:", apps['Installs'].head())

Installs dtype: int64
Sample values: 0    1000000
1     100000
2    1000000
3    1000000
4     500000
Name: Installs, dtype: int64


In [12]:
def convert_size(size):
    if pd.isna(size) or size == 'Varies with device':
        return np.nan
    elif 'M' in str(size):
        return float(str(size).replace('M', ''))
    elif 'k' in str(size):
        return float(str(size).replace('k', '')) / 1024  #convert KB to MB
    else:
        return np.nan
apps['Size'] = apps['Size'].apply(convert_size)
print("Size dtype:", apps['Size'].dtype)
print("Sample values:", apps['Size'].head(10))
print("Missing sizes:", apps['Size'].isnull().sum())

Size dtype: float64
Sample values: 0    31.000000
1     0.088867
2    18.000000
3    33.000000
4    22.000000
5          NaN
6     5.500000
7    10.000000
8          NaN
9    38.000000
Name: Size, dtype: float64
Missing sizes: 1227


In [13]:
apps['Price'] = apps['Price'].str.replace('$', '', regex=False)
apps['Price'] = pd.to_numeric(apps['Price'], errors='coerce')
print("Price dtype:", apps['Price'].dtype)
print("Sample values:", apps['Price'].head())

Price dtype: float64
Sample values: 0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: Price, dtype: float64


In [14]:
# Rating has 1474 missing values
# Also some ratings > 5.0 (errors) — remove those
# Remove ratings that are wrong (>5 is impossible)
apps.loc[apps['Rating'] > 5, 'Rating'] = np.nan
# Fill missing ratings with median 
median_rating = apps['Rating'].median()
apps['Rating'] = apps['Rating'].fillna(median_rating)
print("Rating dtype:", apps['Rating'].dtype)
print("Median rating used for fill:", round(median_rating, 2))
print("Missing ratings now:", apps['Rating'].isnull().sum())

Rating dtype: float64
Median rating used for fill: 4.3
Missing ratings now: 0


In [15]:
apps['Current Ver'] = apps['Current Ver'].fillna('Unknown')
apps['Android Ver'] = apps['Android Ver'].fillna('Unknown')
apps['Size'] = apps['Size'].fillna(apps['Size'].median())
print("Remaining missing values:")
print(apps.isnull().sum())

Remaining missing values:
App               0
Category          0
Rating            0
Reviews           0
Size              0
Installs          0
Type              1
Price             0
Content Rating    0
Genres            0
Last Updated      0
Current Ver       0
Android Ver       0
dtype: int64


In [16]:
apps['is_free'] = apps['Type'].apply(lambda x: True if x == 'Free' else False)
def rating_group(r):
    if r >= 4.5:
        return 'Excellent'
    elif r >= 4.0:
        return 'Good'
    elif r >= 3.0:
        return 'Average'
    else:
        return 'Poor'

apps['rating_group'] = apps['Rating'].apply(rating_group)
def install_bucket(i):
    if i < 1000:
        return 'Under 1K'
    elif i < 10000:
        return '1K - 10K'
    elif i < 100000:
        return '10K - 100K'
    elif i < 1000000:
        return '100K - 1M'
    elif i < 10000000:
        return '1M - 10M'
    else:
        return '10M+'
apps['install_bucket'] = apps['Installs'].apply(install_bucket)
apps['Last Updated'] = pd.to_datetime(apps['Last Updated'], errors='coerce')
apps['update_year'] = apps['Last Updated'].dt.year
print("New columns added:")
print(apps[['App', 'is_free', 'rating_group', 'install_bucket', 'update_year']].head(10))

New columns added:
                                                 App  is_free rating_group  \
0                             GollerCepte Live Score     True         Good   
1                       Ad Block REMOVER - NEED ROOT     True      Average   
2                                SnipSnap Coupon App     True         Good   
3                  US Open Tennis Championships 2018     True         Good   
4                                         DreamTrips     True    Excellent   
5   Adult Color by Number Book - Paint Mandala Pages     True         Good   
6                     BSPlayer ARMv7 VFP CPU support     True         Good   
7  Easy Resume Builder, Resume help, Curriculum v...     True         Good   
8                                  MegaFon Dashboard     True      Average   
9  Buff Thun - Daily Free Webtoon / Comics / Web ...     True    Excellent   

  install_bucket  update_year  
0       1M - 10M         2018  
1      100K - 1M         2013  
2       1M - 10M         2

In [17]:

print("Final dataset shape:", apps.shape)
print("\nColumn dtypes:")
print(apps.dtypes)
print("\nMissing values:")
print(apps.isnull().sum())
apps.to_csv('../output/playstore_clean.csv', index=False)
print("\n✅ Cleaned data saved to output/playstore_clean.csv")

Final dataset shape: (9659, 17)

Column dtypes:
App                          str
Category                     str
Rating                   float64
Reviews                    int64
Size                     float64
Installs                   int64
Type                         str
Price                    float64
Content Rating               str
Genres                       str
Last Updated      datetime64[us]
Current Ver                  str
Android Ver                  str
is_free                     bool
rating_group                 str
install_bucket               str
update_year                int32
dtype: object

Missing values:
App               0
Category          0
Rating            0
Reviews           0
Size              0
Installs          0
Type              1
Price             0
Content Rating    0
Genres            0
Last Updated      0
Current Ver       0
Android Ver       0
is_free           0
rating_group      0
install_bucket    0
update_year       0
dtype: int64

✅ Clea